# Return Estimation from Cointegrated Spreads

This notebook estimates the expected return vectors ($\mu$) and covariance matrices ($\Sigma$) that feed the mean-variance portfolio optimisation in notebook 04. Two parallel estimation approaches are constructed from the in-sample prices (January 2018 -- December 2023) and the three cointegrated pairs identified in `01_spread_exploration.ipynb` --- GS/MS, KO/PEP, and DAL/UAL.

**Pipeline overview.**

| Step | Function | Description |
|------|----------|-------------|
| 1 | `build_spread_return_matrix()` | Compute daily spread log-return differentials for each cointegrated pair |
| 2 | `build_ou_implied_returns()` | Estimate expected one-step spread change from a rolling OU model, annualised |
| 3 | `historical_mean_return()` | Traditional sample-mean estimator for individual asset returns |
| 4 | `shrinkage_covariance()` | Ledoit-Wolf shrinkage covariance to reduce estimation error |
| 5 | Compare and validate | Side-by-side comparison of OU-implied vs historical estimates |

**Spread return construction.**  Daily spread returns are computed as log-return differentials:

$$r^{\text{spread}}_t = \Delta\ln P^y_t - \hat{\beta}\,\Delta\ln P^x_t$$

using log-returns (rather than arithmetic percentage changes) to ensure exact unit consistency with the log-price spread $S_t = \ln P^y_t - \hat{\beta}\ln P^x_t - \hat{\alpha}$ used throughout the cointegration analysis.

**OU-implied return estimation.**  `build_ou_implied_returns()` fits an Ornstein-Uhlenbeck model to each log-price spread over a 60-day rolling window. The expected one-step change in the spread is:

$$E[\Delta S_t] = (\theta - S_t)(1 - e^{-\kappa}), \quad \kappa = \frac{\ln 2}{\text{half-life}}$$

where $\theta$ is the 60-day rolling mean. Since $\Delta S_t \equiv r^{\text{spread}}_t$ in log-price space, $E[\Delta S_t]$ is directly interpretable as a daily log-return for the spread position --- no additional normalisation is required.

**Historical benchmark.**  `historical_mean_return()` computes annualised sample-mean log-returns for each constituent asset over the IS period, providing a traditional $\mu$ vector for mean-variance optimisation.

**Covariance estimation.**  `shrinkage_covariance()` applies Ledoit-Wolf shrinkage (`LedoitWolf` from sklearn), which regularises the sample covariance toward a scaled identity matrix --- ensuring positive definiteness and reducing estimation error in the small-$n$ regime (3 spreads, 6 assets).

**Motivation for comparison.**  The contrast between OU-implied spread returns and historical asset returns motivates the portfolio backtest in notebook 04, where both approaches are evaluated out-of-sample.

**Outputs.**  All $\mu$ vectors, covariance matrices, and return series are saved to `data/processed/` for use in `04_backtest_results.ipynb`.

## Imports

In [1]:
import sys
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore, compute_rolling_half_life, compute_rolling_zscore # spread analysis
from src.modelling import compute_spread_returns, build_spread_return_matrix, historical_mean_return, ewma_mean_return, ou_implied_spread_return, build_ou_implied_returns, sample_covariance, shrinkage_covariance, spread_vs_asset_estimates

## Load in-sample prices (Jan 2018 -- Dec 2023) and cointegrated pairs

The IS period provides the training data for all estimators. Cointegrated pairs were identified in notebook 01 via the Engle-Granger procedure.

In [2]:
# Raw prices
is_prices_df = pd.read_csv('../../data/processed/is_prices_df.csv', index_col=0, parse_dates=True)

# Cointegrated pairs
cointegrated_pairs = pd.read_csv("../../data/processed/cointegrated_pairs.csv")

In [3]:
display(is_prices_df)

,NVDA.O,AMD.O,TSM.N,INTC.O,KO.N,PEP.O,COST.O,TGT.N,JPM.N,BAC.N,...,DAL.N,UAL.O,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N,MRK.N,ABBV.N
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,4.98375,10.98,41.02,46.85,45.54,118.06,179.432718,67.63,107.95,29.90,...,56.74,68.94,59.4505,85.95,181.42,53.6605,139.23,34.543262,53.607232,98.41
2018-01-03,5.31175,11.55,41.71,45.26,45.44,117.75,181.586063,67.17,108.06,29.80,...,55.69,68.49,60.2100,86.35,184.67,54.5760,140.56,34.799208,53.530950,99.95
2018-01-04,5.33975,12.12,41.49,44.43,46.08,118.33,180.175907,65.85,109.04,30.19,...,55.69,69.26,60.4795,87.11,184.33,54.7880,140.55,34.875044,54.398658,99.38
2018-01-05,5.38500,11.88,42.46,44.74,46.07,118.67,178.889617,66.55,108.34,30.33,...,55.97,69.36,61.4570,88.19,186.85,55.5145,141.71,34.941400,54.341447,101.11
2018-01-08,5.55000,12.28,42.44,44.74,46.00,117.99,179.585167,67.18,108.50,30.12,...,54.68,68.51,62.3435,88.28,188.28,55.7105,141.89,34.552741,54.026783,99.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,48.83000,139.60,103.15,48.00,58.32,167.68,656.666974,140.20,167.40,33.43,...,41.13,42.55,153.4200,374.58,353.39,141.4900,155.46,28.400000,107.700000,154.94
2023-12-26,49.27900,143.41,104.45,50.50,58.56,168.86,659.619824,141.03,168.39,33.86,...,40.76,42.08,153.4100,374.66,354.83,141.5200,156.14,28.410000,107.630000,154.62
2023-12-27,49.41700,146.07,104.65,50.76,58.71,169.40,666.800000,142.38,169.40,33.84,...,40.59,41.73,153.3400,374.07,357.83,140.3700,156.35,28.610000,107.980000,154.88


In [4]:
display(cointegrated_pairs)

,y,x,hedge_ratio,intercept,adf_stat,p_value,y_is_I1,x_is_I1,is_cointegrated
0,GS.N,MS.N,0.780150,2.350976,-3.666910,0.020199,True,True,True
1,KO.N,PEP.O,0.641464,0.788960,-3.470124,0.035137,True,True,True
2,DAL.N,UAL.O,0.671613,1.067783,-3.460167,0.036094,True,True,True


## Build spread returns

`build_spread_return_matrix()` constructs a DataFrame of daily spread log-return differentials ($r^{\text{spread}}_t = \Delta\ln P^y_t - \hat{\beta}\,\Delta\ln P^x_t$) for each cointegrated pair. These returns are the basis for the spread-based $\mu$ and $\Sigma$ estimates.

In [5]:
spread_returns = build_spread_return_matrix(is_prices_df, cointegrated_pairs)

print(len(spread_returns))
print(spread_returns.shape)
print(spread_returns.describe())

1508
(1508, 3)
       GS.N_vs_MS.N  KO.N_vs_PEP.O  DAL.N_vs_UAL.O
count   1508.000000    1508.000000    1.508000e+03
mean      -0.000027       0.000016    5.976820e-07
std        0.009937       0.008682    1.233542e-02
min       -0.111370      -0.079468   -1.021001e-01
25%       -0.004448      -0.004050   -5.748011e-03
50%       -0.000108       0.000210   -2.928504e-05
75%        0.004975       0.004056    6.005906e-03
max        0.062398       0.060902    7.591411e-02


In [6]:
display(spread_returns)

,GS.N_vs_MS.N,KO.N_vs_PEP.O,DAL.N_vs_UAL.O
Date,,,
2018-01-03,-0.011442,-0.000512,-0.014281
2018-01-04,0.002043,0.010834,-0.007508
2018-01-05,-0.004820,-0.002058,0.004046
2018-01-08,-0.011387,0.002166,-0.015036
2018-01-09,0.002399,0.004498,-0.009077
...,...,...,...
2023-12-22,0.002065,0.002953,0.000717
2023-12-26,-0.001103,-0.000392,-0.001577
2023-12-27,0.000632,0.000510,0.001430


Check for NaN values in spread returns (none expected after dropping the first observation).

In [7]:
print(spread_returns.isnull().sum())

GS.N_vs_MS.N      0
KO.N_vs_PEP.O     0
DAL.N_vs_UAL.O    0
dtype: int64


## Build OU-implied expected returns

`build_ou_implied_returns()` fits a rolling OU model (window = 60 days) to each log-price spread and computes the annualised expected one-step spread change. This produces the spread-based $\mu$ vector for portfolio optimisation.

In [8]:
ou_mu = build_ou_implied_returns(is_prices_df, cointegrated_pairs, window=60, annualise=True, periods_per_year=252)

print(ou_mu)

GS.N_vs_MS.N     -0.095406
KO.N_vs_PEP.O    -0.086152
DAL.N_vs_UAL.O   -0.237565
Name: ou_implied_return, dtype: float64


## Compute historical mean baseline

`historical_mean_return()` provides the traditional sample-mean estimator for individual asset returns, annualised over the IS period. This serves as a benchmark $\mu$ vector against the OU-implied estimates.

In [9]:
tickers = list(dict.fromkeys(
    cointegrated_pairs['y'].tolist() + cointegrated_pairs['x'].tolist()
))
asset_returns = is_prices_df[tickers].pct_change().dropna()

hist_mu = historical_mean_return(asset_returns, annualise=True, periods_per_year=252)

print(hist_mu)

GS.N     0.119185
KO.N     0.064271
DAL.N    0.045025
MS.N     0.154789
PEP.O    0.083327
UAL.O    0.070081
dtype: float64


In [10]:
def _pair_display_label(y: str, x: str) -> str:
    """Short label for charts, e.g. KO.N / PEP.O → KO/PEP."""
    return f"{y.split('.')[0]}/{x.split('.')[0]}"


def pair_meta_ordered(coint_df: pd.DataFrame, candidates: list) -> list:
    """Cointegrated pairs ordered like ``CANDIDATE_PAIRS``; append any extra rows."""
    meta: list = []
    seen: set[str] = set()
    for y, x in candidates:
        mask = (coint_df["y"] == y) & (coint_df["x"] == x)
        if not mask.any():
            continue
        row = coint_df.loc[mask].iloc[0]
        key = f"{row['y']}_vs_{row['x']}"
        if key in seen:
            continue
        seen.add(key)
        meta.append(
            {
                "spread": key,
                "y": row["y"],
                "x": row["x"],
                "label": _pair_display_label(row["y"], row["x"]),
            }
        )
    for _, row in coint_df.iterrows():
        key = f"{row['y']}_vs_{row['x']}"
        if key not in seen:
            seen.add(key)
            meta.append(
                {
                    "spread": key,
                    "y": row["y"],
                    "x": row["x"],
                    "label": _pair_display_label(row["y"], row["x"]),
                }
            )
    return meta


# Used by all pair plots below (order follows CANDIDATE_PAIRS, then any extras)
PAIR_META = pair_meta_ordered(cointegrated_pairs, CANDIDATE_PAIRS)


## Estimate covariance by computing covariance matrices

In [11]:
spread_cov = shrinkage_covariance(spread_returns, annualise=True)
asset_cov = shrinkage_covariance(asset_returns, annualise=True)

print(spread_cov)
print(asset_cov)

                GS.N_vs_MS.N  KO.N_vs_PEP.O  DAL.N_vs_UAL.O
GS.N_vs_MS.N        0.025146       0.001539        0.001083
KO.N_vs_PEP.O       0.001539       0.019913        0.005078
DAL.N_vs_UAL.O      0.001083       0.005078        0.037109
           GS.N      KO.N     DAL.N      MS.N     PEP.O     UAL.O
GS.N   0.101593  0.028895  0.081523  0.092361  0.028001  0.097082
KO.N   0.028895  0.043819  0.037650  0.032142  0.031805  0.039841
DAL.N  0.081523  0.037650  0.201822  0.091194  0.024379  0.221875
MS.N   0.092361  0.032142  0.091194  0.116515  0.031632  0.106608
PEP.O  0.028001  0.031805  0.024379  0.031632  0.046634  0.023844
UAL.O  0.097082  0.039841  0.221875  0.106608  0.023844  0.305940


## Sanity check - compare vectors side by side

In [12]:
comparison = pd.DataFrame({
    'ou_implied': ou_mu,
    'hist_mean': hist_mu
})
display(comparison)


,ou_implied,hist_mean
DAL.N,NaN,0.045025
DAL.N_vs_UAL.O,-0.237565,NaN
GS.N,NaN,0.119185
GS.N_vs_MS.N,-0.095406,NaN
KO.N,NaN,0.064271
KO.N_vs_PEP.O,-0.086152,NaN
MS.N,NaN,0.154789
PEP.O,NaN,0.083327
UAL.O,NaN,0.070081


### Verify covariance matrices are positive definite

In [13]:
np.linalg.eigvalsh(spread_cov)  # all eigenvalues should be > 0

array([0.01831188, 0.02520342, 0.03865296])

In [14]:
np.linalg.eigvalsh(asset_cov)

array([0.01257431, 0.01630483, 0.02515088, 0.05209359, 0.11946473,
       0.59073481])

### Check end of IN-SAMPLE z-score

In [15]:
# Z-scores must be computed from log-price spreads to match the OU model,
# which now also operates internally on log prices.
prices_log_is = np.log(is_prices_df)

for _, row in cointegrated_pairs.iterrows():
    spread = compute_spread(prices_log_is[row['y']], prices_log_is[row['x']], row['hedge_ratio'], row['intercept'])
    z = compute_zscore(spread, window=60).iloc[-1]
    key = f"{row['y']}_vs_{row['x']}"
    print(f"{key}: z = {z:.3f}, ou_mu = {ou_mu[key]:.4f}")

GS.N_vs_MS.N: z = 0.675, ou_mu = -0.0954
KO.N_vs_PEP.O: z = 0.809, ou_mu = -0.0862
DAL.N_vs_UAL.O: z = 1.315, ou_mu = -0.2376


## Visualise Table

In [16]:
def _signal_from_z(z: float) -> str:
    if z > 1.0:
        return "Spread well above mean"
    if z < -1.0:
        return "Spread well below mean"
    if z > 0.25:
        return "Spread above mean"
    if z < -0.25:
        return "Spread below mean"
    return "Near mean"


val_rows = []
prices_log_is = np.log(is_prices_df)  # use log prices consistent with OU model

for pm in PAIR_META:
    crow = cointegrated_pairs[
        (cointegrated_pairs["y"] == pm["y"]) & (cointegrated_pairs["x"] == pm["x"])
    ].iloc[0]
    spread = compute_spread(
        prices_log_is[crow["y"]],
        prices_log_is[crow["x"]],
        crow["hedge_ratio"],
        crow["intercept"],
    )
    z_end = float(compute_zscore(spread, window=60).iloc[-1])
    ou = float(ou_mu[pm["spread"]])
    sig = _signal_from_z(z_end)
    # Consistent: positive z → spread falls → negative OU return (z * ou < 0)
    # Near-zero z → both signals are weak → "Yes" (no contradiction)
    consistent = (
        "Yes"
        if abs(z_end) < 0.5
        else ("Yes" if z_end * ou < 0 else "Review")
    )
    val_rows.append(
        {
            "Pair": pm["label"],
            "End-of-IS Z-Score": z_end,
            "OU-Implied Return (ann.)": ou,
            "Signal": sig,
            "Consistent": consistent,
        }
    )

validation_df = pd.DataFrame(val_rows)

display(validation_df)

,Pair,End-of-IS Z-Score,OU-Implied Return (ann.),Signal,Consistent
0,KO/PEP,0.808854,-0.086152,Spread above mean,Yes
1,GS/MS,0.675478,-0.095406,Spread above mean,Yes
2,DAL/UAL,1.314954,-0.237565,Spread well above mean,Yes


In [17]:
print(validation_df.to_latex(index=False))

\begin{tabular}{lrrll}
\toprule
Pair & End-of-IS Z-Score & OU-Implied Return (ann.) & Signal & Consistent \\
\midrule
KO/PEP & 0.808854 & -0.086152 & Spread above mean & Yes \\
GS/MS & 0.675478 & -0.095406 & Spread above mean & Yes \\
DAL/UAL & 1.314954 & -0.237565 & Spread well above mean & Yes \\
\bottomrule
\end{tabular}



## Visualise Results

### Cumulative spread returns vs constituent assets

In [18]:
_n_cum = len(PAIR_META)
fig1 = make_subplots(
    rows=_n_cum,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[p["label"] for p in PAIR_META],
    vertical_spacing=0.08,
)

for i, pm in enumerate(PAIR_META, start=1):
    cum_spread = (1 + spread_returns[pm["spread"]]).cumprod()
    cum_y      = (1 + asset_returns[pm["y"]]).cumprod()
    cum_x      = (1 + asset_returns[pm["x"]]).cumprod()

    fig1.add_trace(go.Scatter(
        x=cum_spread.index, y=cum_spread.values,
        name=f"{pm['label']} Spread",
        line=dict(width=1.8), showlegend=(i == 1),
        legendgroup="spread",
    ), row=i, col=1)
    fig1.add_trace(go.Scatter(
        x=cum_y.index, y=cum_y.values,
        name=pm["y"],
        line=dict(width=1.2, dash="dot"), showlegend=True,
        legendgroup=pm["y"],
    ), row=i, col=1)
    fig1.add_trace(go.Scatter(
        x=cum_x.index, y=cum_x.values,
        name=pm["x"],
        line=dict(width=1.2, dash="dash"), showlegend=True,
        legendgroup=pm["x"],
    ), row=i, col=1)

    fig1.add_hline(
        y=1, line=dict(color="gray", dash="dash", width=0.8),
        annotation_text="Base" if i == 1 else None,
        annotation_font_size=9,
        annotation_position="right",
        row=i, col=1,
    )

fig1.update_yaxes(title_text="Cumulative Return", tickformat=".2f")
fig1.update_xaxes(title_text="Date", row=_n_cum, col=1)
fig1.update_layout(
    title="Cumulative Returns: Cointegrated Spread vs Constituent Assets (IS 2019\u20132023)",
    height=350 * max(_n_cum, 1),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    colorway=[
        "#1f77b4", "#d62728",
        "#9467bd", "#8c564b",
        "#17becf", "#bcbd22",
    ],
)
fig1.show()

#### Save to Matplotlib

In [19]:
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import scienceplots
plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
})

n_pairs = len(PAIR_META)
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(6.5, 2.17 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, pm, label, color in zip(axes, PAIR_META, panel_labels, prop_cycle):
    first = (label == panel_labels[0])

    cum_s = (1 + spread_returns[pm["spread"]]).cumprod()
    cum_y = (1 + asset_returns[pm["y"]]).cumprod()
    cum_x = (1 + asset_returns[pm["x"]]).cumprod()

    ax.plot(cum_s.index, cum_s.values, linewidth=1.8, color=color,
            label=r"Spread" if first else None)
    ax.plot(cum_y.index, cum_y.values, linewidth=1.0, linestyle="--",
            label=r"Constituent $Y$" if first else None)
    ax.plot(cum_x.index, cum_x.values, linewidth=1.0, linestyle=":",
            label=r"Constituent $X$" if first else None)
    ax.axhline(1, color="grey", linewidth=0.6, linestyle="--",
               label=r"Base" if first else None)

    ax.set_ylabel(r"Cum.\ Return")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    y_name = TICKER_NAMES.get(pm["y"], pm["y"])
    x_name = TICKER_NAMES.get(pm["x"], pm["x"])
    ax.set_title(f"{label} {y_name}/{x_name}", loc="left", fontsize=10, fontweight="bold")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/03_cumulative_returns.pdf", bbox_inches="tight")
plt.close(fig)

### OU-implied vs historical $\mu$ bar chart

In [20]:
pair_labels = [p["label"] for p in PAIR_META]
ou_mu_ordered = ou_mu.reindex([p["spread"] for p in PAIR_META])
hist_tickers = hist_mu.index.tolist()

fig_mu = make_subplots(
    rows=1, cols=2,
    subplot_titles=["OU-Implied \u03bc (by Pair)", "Historical Mean \u03bc (by Ticker)"],
    horizontal_spacing=0.12,
)

fig_mu.add_trace(go.Bar(
    x=pair_labels,
    y=ou_mu_ordered.values,
    name="OU-Implied",
    showlegend=True,
), row=1, col=1)

fig_mu.add_trace(go.Bar(
    x=hist_tickers,
    y=hist_mu.values,
    name="Historical Mean",
    showlegend=True,
), row=1, col=2)

fig_mu.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=1, col=1)
fig_mu.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=1, col=2)

fig_mu.update_yaxes(title_text="Expected Return (ann.)", tickformat=".1%", row=1, col=1)
fig_mu.update_yaxes(tickformat=".1%", row=1, col=2)
fig_mu.update_xaxes(tickangle=-45, row=1, col=1)
fig_mu.update_layout(
    title="Expected Return Estimates (Annualised, IS 2019\u20132023)",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_mu.show()

#### Save to Matplotlib

In [21]:
from src.modelling import compute_spread, compute_half_life

rows = []
for _, row in cointegrated_pairs.iterrows():
    label     = f"{row['y'].split('.')[0]}/{row['x'].split('.')[0]}"
    spread    = compute_spread(
        is_prices_df[row["y"]], is_prices_df[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()

    hl        = compute_half_life(spread)
    kappa     = np.log(2) / hl
    theta     = spread.rolling(window=60).mean().iloc[-1]
    s_t       = spread.iloc[-1]
    exp_delta = (theta - s_t) * (1 - np.exp(-kappa))
    daily_ret = exp_delta / abs(s_t) if s_t != 0 else np.nan
    ann_ret   = daily_ret * 252

    rows.append({
        "Pair":               label,
        "S_t (spread level)": round(s_t, 6),
        "θ (rolling mean)":   round(theta, 6),
        "θ − S_t":            round(theta - s_t, 6),
        "κ (speed)":          round(kappa, 4),
        "E[ΔS] (daily)":      round(exp_delta, 6),
        "|S_t| (divisor)":    round(abs(s_t), 6),
        "Daily return":       f"{daily_ret:.4%}",
        "Ann. OU return":     f"{ann_ret:.2%}",
    })

decomp_df = pd.DataFrame(rows).set_index("Pair")
display(decomp_df)

print("\nKey insight:")
for _, row in decomp_df.iterrows():
    s = float(row["S_t (spread level)"])
    ann = row["Ann. OU return"]
    if abs(s) < 0.05:
        print(f"  {row.name}: |S_t| = {s:.6f} — near zero → extreme annualised return ({ann})")
    else:
        print(f"  {row.name}: |S_t| = {s:.6f} — well away from zero → moderate return ({ann})")

,S_t (spread level),θ (rolling mean),θ − S_t,κ (speed),E[ΔS] (daily),|S_t| (divisor),Daily return,Ann. OU return
Pair,,,,,,,,
GS/MS,310.669996,270.099840,-40.570156,0.0022,-0.088261,310.669996,-0.0284%,-7.16%
KO/PEP,-50.805151,-50.203236,0.601915,0.0030,0.001782,50.805151,0.0035%,0.88%
DAL/UAL,11.451474,8.866268,-2.585206,0.0046,-0.011907,11.451474,-0.1040%,-26.20%



Key insight:
  GS/MS: |S_t| = 310.669996 — well away from zero → moderate return (-7.16%)
  KO/PEP: |S_t| = -50.805151 — well away from zero → moderate return (0.88%)
  DAL/UAL: |S_t| = 11.451474 — well away from zero → moderate return (-26.20%)


In [22]:
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
})
pair_labels = [p["label"] for p in PAIR_META]
ou_mu_ordered = ou_mu.reindex([p["spread"] for p in PAIR_META])
hist_tickers = hist_mu.index.tolist()
hist_labels  = [TICKER_NAMES.get(t, t) for t in hist_tickers]
prop_cycle   = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.5, 2.20))

ax1.bar(pair_labels, ou_mu_ordered.values, color=prop_cycle[0], width=0.4)
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax1.tick_params(axis="x", rotation=45)
ax1.set_title("(a)", loc="left", fontsize=10, fontweight="bold")
ax1.set_ylabel(r"Expected Return (ann.)")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2.bar(hist_labels, hist_mu.values, color=prop_cycle[1], width=0.4)
ax2.tick_params(axis="x", rotation=45)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_title("(b)", loc="left", fontsize=10, fontweight="bold")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

fig.suptitle(r"Expected Return Estimates (Annualised, IS 2019--2023)", y=1.01)
fig.tight_layout()
fig.savefig("figures/03_mu_comparison.pdf", bbox_inches="tight")
plt.close(fig)

### Covariance/correlaton heatmap

In [23]:
import plotly.figure_factory as ff

corr = asset_cov / np.outer(np.sqrt(np.diag(asset_cov)), np.sqrt(np.diag(asset_cov)))
tickers_ordered = list(dict.fromkeys(
    [t for _, row in cointegrated_pairs.iterrows() for t in (row["y"], row["x"])]
))

corr_df = pd.DataFrame(corr, index=tickers_ordered, columns=tickers_ordered)

fig_corr = go.Figure(go.Heatmap(
    z=corr_df.values,
    x=corr_df.columns.tolist(),
    y=corr_df.index.tolist(),
    colorscale="RdBu_r",
    zmid=0, zmin=-1, zmax=1,
    text=np.round(corr_df.values, 2),
    texttemplate="%{text}",
    textfont=dict(size=11),
    colorbar=dict(title="Correlation"),
))
fig_corr.update_layout(
    title="Asset Return Correlation Matrix (IS 2019\u20132023, Shrinkage Covariance)",
    template="plotly_white",
    height=500,
    width=550,
    xaxis=dict(side="bottom"),
)
fig_corr.show()

#### Save to Matplotlib

In [24]:
corr = asset_cov / np.outer(np.sqrt(np.diag(asset_cov)), np.sqrt(np.diag(asset_cov)))
corr_df = pd.DataFrame(corr, index=tickers_ordered, columns=tickers_ordered)

fig, ax = plt.subplots(figsize=(5.85, 5.04))
im = ax.imshow(corr_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(tickers_ordered)))
ax.set_yticks(range(len(tickers_ordered)))
friendly_tickers = [TICKER_NAMES.get(t, t) for t in tickers_ordered]
ax.set_xticklabels(friendly_tickers, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(friendly_tickers, fontsize=9)

for i in range(len(tickers_ordered)):
    for j in range(len(tickers_ordered)):
        ax.text(j, i, f"{corr_df.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if abs(corr_df.values[i, j]) > 0.6 else "black")

fig.colorbar(im, ax=ax, label="Correlation")
fig.tight_layout()
fig.savefig("figures/03_correlation_heatmap.pdf", bbox_inches="tight")
plt.close(fig)


## Save outputs

In [25]:
ou_mu.to_csv("../../data/processed/spread_mu.csv")
hist_mu.to_csv("../../data/processed//asset_mu.csv")
spread_cov.to_csv("../../data/processed/spread_cov.csv")
asset_cov.to_csv("../../data/processed/asset_cov.csv")
spread_returns.to_csv("../../data/processed/spread_returns.csv")
asset_returns.to_csv("../../data/processed/asset_returns.csv")
